In [24]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

print(f'pandas : {pd.__version__}')
print(f'requests: {requests.__version__}')

All libraries imported successfully!
pandas : 2.2.2
requests: 2.32.4


ETL stands for Extract, Transform and Load


It is a 3 step data integration pattern




PART 1 : ETL on messy sales data

Activity 1 - Clean messy_sales_data.csv

Upload messy_sales_data.csv to Colab files panel , then run the cells below

**Known data quality issues in the file:**

    1.Missing values in customer_name, quantity,category
    2.Duplicate rows
    3.Mixed data formats: YYYY--MM--DD and DD-MM-YYYY
    4.Inconsistent text case in customer_name
    5.Wrong category values

In [25]:
df=pd.read_csv('messy_sales_data.csv')
print(f"Dataset loaded:{df.shape[0]} rows,{df.shape[1]} columns")
print(f"Columns:{df.columns.tolist()}")
df.head()

Dataset loaded:30 rows,9 columns
Columns:['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [26]:
print("="*55)
print("DATA QUALITY DIALYSIS REPORT")
print("="*55)

print("\n[1] Missing Values per column")
print(df.isnull().sum())

print(f"\n[2] Duplicate Rows: {df.duplicated().sum()}")

print("\n[3] Data Types:")
print(df.dtypes)
print("\n[4] Unique Categories:", df['category'].unique())
print("\n[5] Sample Customer Name:", df['customer_name'].dropna().unique()[:5])
print('\n[6] Sample order_date values:', df['order_date'].unique()[:5])

DATA QUALITY DIALYSIS REPORT

[1] Missing Values per column
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] Duplicate Rows: 0

[3] Data Types:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] Unique Categories: ['Electronics' 'Accessories' nan]

[5] Sample Customer Name: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta']

[6] Sample order_date values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024']


In [27]:
df1=df.copy()
print(f"Working copy created:{df.shape}")
print("df is untouched - we can always reset by running df1= df.copy()")

Working copy created:(30, 9)
df is untouched - we can always reset by running df1= df.copy()


In [28]:
print('Before fixing nulls:',df.isnull().sum().sum(),'total missing values')
df['customer_name'].fillna('Unkonwn Customer',inplace=True)
median_qty=df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)
print(f' Filled missing quantity with median: {median_qty}')
df['category'].fillna('Uncategorized',inplace=True)
print('After fixing nulls:',df.isnull().sum().sum(),'total missing values')

Before fixing nulls: 7 total missing values
 Filled missing quantity with median: 2.0
After fixing nulls: 1 total missing values


In [29]:
print(f"Before deduplication:{len(df1)} rows")
print(f'duplicate rows:{df1.duplicated().sum()}')

print('\nDuplicate rows:')

print(df1[df1.duplicated(keep=False)][['order_id','customer_name','product','order_date']])

initial_df1_length = len(df1)
df1.drop_duplicates(inplace=True)
print(f'\nAfter deduplication:{len(df1)} rows')
print(f'Rows removed:{initial_df1_length - len(df1)}')

Before deduplication:30 rows
duplicate rows:0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

After deduplication:30 rows
Rows removed:0


In [30]:
print('Sample data before paesing:')
print(df1['order_date'].head(8).tolist())

df1['order_date']=pd.to_datetime(
    df1['order_date'],
    dayfirst=False,
    errors='coerce'
    )

nat_count=df1['order_date'].isnull().sum()
print(f'\nUnparsable dates(NaT):{nat_count}')

df1['year']=df1['order_date'].dt.year
df1['month']=df1['order_date'].dt.month
df1['month_name']=df1['order_date'].dt.strftime('%B')

print('\nSample dates after parsing:')
print(df1[['order_date','year','month','month_name']].head(5))



Sample data before paesing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

Unparsable dates(NaT):2

Sample dates after parsing:
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January


In [31]:
print('Before Standardization:',df1['customer_name'].unique()[:6])

df1['customer_name']=(
    df1['customer_name']
    .str.strip()  #Removes leading whitesqapces
    .str.title()  #Convert to title case
)

print('After Standardization:',df1['customer_name'].unique()[:6])
print(f'\nBefore: Keyboard rows with Electronics category:')
wrong_mask=(df1['product']=='keyboard') & (df1['category']=='Electronics')
print(df1[wrong_mask][['product','category']])

df1.loc[wrong_mask,'category']='Accessories'
print('After fix - Unique categories:', df1['category'].unique())

Before Standardization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After Standardization: ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']

Before: Keyboard rows with Electronics category:
Empty DataFrame
Columns: [product, category]
Index: []
After fix - Unique categories: ['Electronics' 'Accessories' nan]


In [32]:
df1['quantity']=pd.to_numeric(df1['quantity'],errors='coerce')
df1['unit_price']=pd.to_numeric(df1['unit_price'],errors='coerce')
df1['revenue']=df1['quantity']*df1['unit_price']
print('Revenue column created:')
print(df1[['customer_name','product','quantity','unit_price','revenue']].head(5))
print(f'\nTotal Revenue across all orders: Rs{df1["revenue"].sum():,.0f}')

Revenue column created:
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop       2.0       45000  90000.0
1    Priya Nair       NaN       1.0       15000  15000.0
2    Amit Verma  Keyboard       3.0        1200   3600.0
3  Sunita Patel   Monitor       NaN       22000      NaN
4  Ramesh Kumar    Laptop       2.0       45000  90000.0

Total Revenue across all orders: Rs679,000


In [33]:
print('='*55)
print('POST-CLEANING VALIDATION REPORT')
print('='*55)
print(f"Original rows:{len(df)}")
print(f"Cleaned rows:{len(df1)}")
print(f"Rows removed: {len(df)}-{len(df1)} (duplicates)")
print(f"Missing values:{df1.isnull().sum().sum()}")
print(f"Dupicates:{df1.duplicated().sum()}")
print(f"Date nulls:{df1['order_date'].isnull().sum()}")
print(f"Revenue NaN:{df1['revenue'].isnull().sum()}")
print(f"Categories:{sorted(df1['category'].dropna().unique())}")
print('='*55)

all_clean=(
    df1.isnull().sum().sum()==0 and
    df1.duplicated().sum()==0
)
print(f'DATA IS CLEAN: {all_clean}')


POST-CLEANING VALIDATION REPORT
Original rows:30
Cleaned rows:30
Rows removed: 30-30 (duplicates)
Missing values:18
Dupicates:0
Date nulls:2
Revenue NaN:3
Categories:['Accessories', 'Electronics']
DATA IS CLEAN: False


In [34]:
product_rev=(
    df1.groupby('product')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue',ascending=False)
)
print('Revenue by Product:')
print(product_rev.to_string(index=False))

Category_summary=df1.groupby('category').agg(
    total_revenue=('revenue','sum'),
    total_order_value=('revenue','mean'),
    num_orders=('order_id','count'),
    unique_products=('product','nunique')
).round(2).reset_index()
print('\n Category Summary:')
print(Category_summary.to_string(index=False))

Revenue by Product:
   product  revenue
    Laptop 450000.0
   Monitor 110000.0
Headphones  28000.0
     Mouse  20800.0
  Keyboard  20400.0
   USB Hub  19800.0
    Webcam  15000.0

 Category Summary:
   category  total_revenue  total_order_value  num_orders  unique_products
Accessories        71200.0            5933.33          13                4
Electronics       563800.0           40271.43          16                4


In [35]:
df.to_csv('sales_data_cleaned.csv',index=False)
print('Cleaned data saved to : sales_data_cleaned.csv')
print(f'Final dataset:{df.shape[0]} rows x {df.shape[1]} columns')
print('\nETL Pipeline for Sales Data : COMPLETE')
print(' EXTRACT   -> messy_sales_data.csv loaded')
print(' TRANSFORM -> nulls fixed, dupes removed, datas parsed, names standardized')
print(' LOAD      -> sales_data_cleaned.csv saved')

Cleaned data saved to : sales_data_cleaned.csv
Final dataset:30 rows x 9 columns

ETL Pipeline for Sales Data : COMPLETE
 EXTRACT   -> messy_sales_data.csv loaded
 TRANSFORM -> nulls fixed, dupes removed, datas parsed, names standardized
 LOAD      -> sales_data_cleaned.csv saved


In [36]:
API_KEY='420832ea5d06fd35a3b722687c824244'
BASE_URL="https://api.openweathermap.org/data/2.5/weather"
CITIES=['Mumbai','chennai','Delhi','Bengaluru','Hyderabad','Pune','Kolkatta','Jaipur']
print(f'API configured for {len(CITIES)} cities ')
print(f'Cities: {CITIES}')
print('\nIMPORTANT: Replace YOUR_API_KEY_HERE with your actual key before running.')

API configured for 8 cities 
Cities: ['Mumbai', 'chennai', 'Delhi', 'Bengaluru', 'Hyderabad', 'Pune', 'Kolkatta', 'Jaipur']

IMPORTANT: Replace YOUR_API_KEY_HERE with your actual key before running.


In [37]:
import requests;
def fetch_weather(city, API_KEY):
    """
    Fetch current weather data for a given city.
    Returns a dictionary with weather metrics, or None on failure.
    """
    params = {
        'q':     city,
        'appid': API_KEY,
        'units': 'metric'
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            return {
                'city':        city,
                'temperature': round(data['main']['temp'], 1),
                'feels_like':  round(data['main']['feels_like'], 1),
                'humidity':    data['main']['humidity'],
                'pressure':    data['main']['pressure'],
                'wind_speed':  data['wind']['speed'],
                'condition':   data['weather'][0]['description'].title(),
                'visibility':  data.get('visibility', 0) // 1000
            }
        else:
            print(f'  ERROR {response.status_code} for {city}: {response.json().get("message","unknown error")}')
            return None

    except requests.exceptions.ConnectionError:
        print(f'  CONNECTION ERROR for {city} — check internet connection')
        return None
    except requests.exceptions.Timeout:
        print(f'  TIMEOUT for {city} — API did not respond in 10 seconds')
        return None

print('Calling Weather API...')
weather_records = []


for city in CITIES:
    print(f'  Fetching: {city}...', end='')
    record = fetch_weather(city, API_KEY)
    if record:
        weather_records.append(record)
        print(f' {record["temperature"]}°C, {record["condition"]}')
    else:
        print(' FAILED')

print(f'\nSuccessfully fetched: {len(weather_records)}/{len(CITIES)} cities')

Calling Weather API...
  Fetching: Mumbai... 32.0°C, Haze
  Fetching: chennai... 32.5°C, Scattered Clouds
  Fetching: Delhi... 30.1°C, Thunderstorm With Light Rain
  Fetching: Bengaluru... 29.5°C, Overcast Clouds
  Fetching: Hyderabad... 36.2°C, Scattered Clouds
  Fetching: Pune... 31.4°C, Clear Sky
  Fetching: Kolkatta...  ERROR 404 for Kolkatta: city not found
 FAILED
  Fetching: Jaipur... 40.6°C, Haze

Successfully fetched: 7/8 cities


In [38]:
if len(weather_records) == 0:
    print('Using fallback weather data (API not available)')
    weather_records = [
        {'city':'Mumbai',    'temperature':32.5,'feels_like':36.0,'humidity':78,'pressure':1009,'wind_speed':5.2,'condition':'Partly Cloudy','visibility':8},
        {'city':'Delhi',     'temperature':38.2,'feels_like':41.0,'humidity':35,'pressure':1002,'wind_speed':3.8,'condition':'Clear Sky',    'visibility':10},
        {'city':'Bangalore', 'temperature':26.1,'feels_like':27.0,'humidity':62,'pressure':1016,'wind_speed':2.5,'condition':'Overcast',     'visibility':7},
        {'city':'Chennai',   'temperature':34.8,'feels_like':39.0,'humidity':72,'pressure':1008,'wind_speed':6.1,'condition':'Hazy',         'visibility':5},
        {'city':'Hyderabad', 'temperature':35.4,'feels_like':38.5,'humidity':45,'pressure':1005,'wind_speed':4.2,'condition':'Clear Sky',    'visibility':10},
        {'city':'Kolkata',   'temperature':33.7,'feels_like':37.8,'humidity':80,'pressure':1007,'wind_speed':4.8,'condition':'Humid',        'visibility':6},
        {'city':'Pune',      'temperature':29.3,'feels_like':31.0,'humidity':55,'pressure':1014,'wind_speed':3.1,'condition':'Partly Cloudy','visibility':9},
        {'city':'Jaipur',    'temperature':40.1,'feels_like':43.0,'humidity':22,'pressure':998, 'wind_speed':5.5,'condition':'Sunny',        'visibility':12},
    ]
    print(f'Fallback data loaded for {len(weather_records)} cities')
else:
    print(f'Using live API data for {len(weather_records)} cities')

Using live API data for 7 cities


In [39]:
weather_df = pd.DataFrame(weather_records)
print('Weather DataFrame created:')
print(weather_df.to_string(index=False))
print(f'\nShape: {weather_df.shape}')
print(f'Missing values: {weather_df.isnull().sum().sum()}')
print(f'\nData types:')
print(weather_df.dtypes)

Weather DataFrame created:
     city  temperature  feels_like  humidity  pressure  wind_speed                    condition  visibility
   Mumbai         32.0        39.0        66      1008        4.63                         Haze           4
  chennai         32.5        39.5        75      1006        7.20             Scattered Clouds           6
    Delhi         30.1        32.5        58       998        8.75 Thunderstorm With Light Rain           4
Bengaluru         29.5        31.4        57      1010        3.11              Overcast Clouds          10
Hyderabad         36.2        37.6        34      1005        3.09             Scattered Clouds           6
     Pune         31.4        32.4        46      1009        6.25                    Clear Sky          10
   Jaipur         40.6        41.0        22      1000        7.20                         Haze           4

Shape: (7, 8)
Missing values: 0

Data types:
city            object
temperature    float64
feels_like     fl

In [40]:
print('=' * 50)
print('  WEATHER ANALYSIS REPORT — 8 INDIAN CITIES')
print('=' * 50)

hottest = weather_df.loc[weather_df['temperature'].idxmax()]
coldest = weather_df.loc[weather_df['temperature'].idxmin()]

print(f"\nHottest city : {hottest['city']} at {hottest['temperature']}°C")
print(f"Coldest city : {coldest['city']} at {coldest['temperature']}°C")
print(f"Most humid   : {weather_df.loc[weather_df['humidity'].idxmax()]['city']} "
      f"({weather_df['humidity'].max()}%)")

print(f"\nAverage temperature : {weather_df['temperature'].mean():.1f}°C")
print(f"Average humidity    : {weather_df['humidity'].mean():.1f}%")
print(f"Average wind speed  : {weather_df['wind_speed'].mean():.1f} m/s")

print('\nCities ranked by temperature (hottest first):')
ranked = weather_df[['city','temperature','humidity']].sort_values('temperature', ascending=False)
print(ranked.to_string(index=False))

  WEATHER ANALYSIS REPORT — 8 INDIAN CITIES

Hottest city : Jaipur at 40.6°C
Coldest city : Bengaluru at 29.5°C
Most humid   : chennai (75%)

Average temperature : 33.2°C
Average humidity    : 51.1%
Average wind speed  : 5.7 m/s

Cities ranked by temperature (hottest first):
     city  temperature  humidity
   Jaipur         40.6        22
Hyderabad         36.2        34
  chennai         32.5        75
   Mumbai         32.0        66
     Pune         31.4        46
    Delhi         30.1        58
Bengaluru         29.5        57


In [41]:
weather_df.to_csv('weather_data.csv', index=False)

print('Weather data saved to: weather_data.csv')
print('\nWeather ETL Pipeline: COMPLETE')
print('  EXTRACT   → OpenWeatherMap API called for 8 cities')
print('  TRANSFORM → JSON parsed, DataFrame built, units converted')
print('  LOAD      → weather_data.csv saved')

Weather data saved to: weather_data.csv

Weather ETL Pipeline: COMPLETE
  EXTRACT   → OpenWeatherMap API called for 8 cities
  TRANSFORM → JSON parsed, DataFrame built, units converted
  LOAD      → weather_data.csv saved


Practise Questions

1.What are the 3 stages of ELT?Explain each stage using an ex from sales dataset

2.A DataFrame has 500 rows. After calling df.dropna(). It has 42 rows . What does this tell you?

3.Write code to remove duplicates from df where 'same row' means same 'customer_name' and 'product'

4.What is the diff btw fillna() and fillna(df['col'].median())?When would you prefer each?

5.Write code to call the weather api for Delhi and print temp in Celsius

6.What does reponse.status_code==200 mean? What should you do when the code is 401?


### Question 1: What are the 3 stages of ELT? Explain each stage using an example from the sales dataset.

ELT stands for **Extract, Load, Transform**.

1.  **Extract:** Reading data from the source
(e.g : `df = pd.read_csv('messy_sales_data.csv')`).

2.  **Load:** Moving the raw extracted data to a target system
 (e.g :Saving the sales data into a system for further processing.
 `df.to_csv('sales_data_cleaned.csv', index=False)`).

3.  **Transform:** The data is cleaned and modified into a useful format.
(e.g :`df['customer_name'].fillna('Unknown')`, `df['order_date'] = pd.to_datetime(df['order_date'])`).

### Question 2: A DataFrame has 500 rows. After calling `df.dropna()`, it has 42 rows. What does this tell you?

It means most of the rows had missing values.
df.dropna() removes rows that contain null or NaN values.

So, out of 500 rows, only 42 rows had complete data and the remaining 458 rows had at least one missing value.

### Question 3: Write code to remove duplicates from `df` where 'same row' means same 'customer_name' and 'product'.

`df.drop_duplicates(subset=['customer_name', 'product'], inplace=True)`

This removes rows where both the customer name and product are repeated.

### Question 4: What is the difference between `fillna()` and `fillna(df['col'].median())`? When would you prefer each?

fillna(value) is used to fill missing values with a fixed value like 0 or "Unknown".

Eg : `df['city'].fillna('Unknown')`
     `fillna(df['col'].median())` fills missing values using the median of that column.

Eg : `df['price'].fillna(df['price'].median())`

We use median mostly for numerical data because it is less affected by outliers.

### Question 5: Write code to call the weather API for Delhi and print temperature in Celsius.

In [42]:
API_KEY = '420832ea5d06fd35a3b722687c824244'
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"
CITY = 'Delhi'

params = {
    'q': CITY,
    'appid': API_KEY,
    'units': 'metric'
}

try:
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    weather_data = response.json()

    if weather_data and 'main' in weather_data and 'temp' in weather_data['main']:
        temperature_celsius = weather_data['main']['temp']
        print(f"Current temperature in {CITY}: {temperature_celsius}°C")
    else:
        print(f"Could not retrieve temperature for {CITY}. Response: {weather_data}")

except requests.exceptions.RequestException as e:
    print(f"Error making API request: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Current temperature in Delhi: 30.05°C


### Question 6: What does `response.status_code == 200` mean? What should you do when the code is `401`?

response.status_code == 200 means the API request was successful.

401 = Unauthorized.
This usually happens when the API key is wrong, expired, or missing.

You should:

Check whether the API key is correct.

Make sure the key is added properly in the request.

Verify that the API key is active.